## Database

### Create DB Schema

In [1]:
import psycopg2

POSTGRES_HOST: str = "localhost"
POSTGRES_PORT: int = 5432
POSTGRES_DBNAME: str = "dms_meta"
POSTGRES_USER: str = "dms"
POSTGRES_PASSWORD: str = "dms"

pg_conn = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=POSTGRES_DBNAME,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
)

In [2]:
from pathlib import Path
current_directory = Path.cwd()

In [ ]:


schema_path = current_directory / "database" / "schemas" / "schema.sql"
if not schema_path.exists():
    raise RuntimeError(f"Schema file not found at {schema_path}")

with pg_conn.cursor() as cur:
    with open(schema_path, "r", encoding="utf-8") as f:
        cur.execute(f.read())

print("Schema ensured (documents, ocr_results)")

Schema ensured (documents, ocr_results)


In [10]:
pg_conn.commit()

In [3]:
with pg_conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = 'documents';
    """)
    columns = cur.fetchall()

for col in columns:
    print(col)


('id', 'uuid')
('file_size', 'bigint')
('created_at', 'timestamp with time zone')
('updated_at', 'timestamp with time zone')
('mime_type', 'character varying')
('document_type', 'character varying')
('linked_entity', 'character varying')
('linked_entity_id', 'character varying')
('hash_sha256', 'character varying')
('text_extraction_status', 'character varying')
('processing_status', 'character varying')
('source_filename', 'character varying')
('blob_path', 'character varying')
('acu_result_blob_path', 'character varying')


In [4]:
from src.dms.adapters import AzureBlobStorageClient, PostgresMetadataRepository
from src.dms.service import DmsService
import os

In [5]:
from azure.storage.blob import BlobServiceClient
connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

CONTAINER_NAME = "documents"

container_client = blob_service_client.get_container_client(CONTAINER_NAME)

try:
    container_client.create_container()
except Exception as e:
    print(e)
    pass  # container likely already exists

The specified container already exists.
RequestId:78db9910-501e-003c-74e8-9bdb26000000
Time:2026-02-12T06:28:55.4701175Z
ErrorCode:ContainerAlreadyExists
Content: <?xml version="1.0" encoding="utf-8"?><Error><Code>ContainerAlreadyExists</Code><Message>The specified container already exists.
RequestId:78db9910-501e-003c-74e8-9bdb26000000
Time:2026-02-12T06:28:55.4701175Z</Message></Error>


In [6]:
storage_client = AzureBlobStorageClient(blob_service_client)
metadata_repo = PostgresMetadataRepository(pg_conn)

dms_service = DmsService(storage_client=storage_client, metadata_repository=metadata_repo)

In [ ]:
from datetime import datetime

# Resolve sample file
sample_pdf_path = current_directory / "data" / "AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf"
assert sample_pdf_path.exists(), "Sample PDF not found"

DOCUMENT_TYPE: str = "license-agreement"

document_id = dms_service.upload_document(
    file_path=sample_pdf_path,
    document_type=DOCUMENT_TYPE,
    source_filename=sample_pdf_path.name,
)

print("Uploaded:", document_id)

metadata = dms_service.get_document(document_id)
print("Metadata keys:", sorted(list(metadata.keys())) if metadata else None)



Uploaded: a4a3ad8a-b3c8-4618-90d5-9d3c69b88bd8
Metadata keys: ['acu_result_blob_path', 'blob_path', 'created_at', 'document_type', 'file_size', 'hash_sha256', 'id', 'linked_entity', 'linked_entity_id', 'mime_type', 'processing_status', 'source_filename', 'text_extraction_status', 'updated_at']


TypeError: DmsService.download_document() takes 1 positional argument but 2 were given

In [8]:
downloaded = dms_service.download_document(document_id=document_id)
print("Downloaded bytes:", len(downloaded) if downloaded else None)

Downloaded bytes: 176502


In [9]:
from psycopg2.extras import RealDictCursor

with pg_conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute("""
        SELECT *
        FROM documents
        ORDER BY created_at DESC;
    """)
    rows = cur.fetchall()

for row in rows:
    print(row)


RealDictRow({'id': 'a4a3ad8a-b3c8-4618-90d5-9d3c69b88bd8', 'source_filename': 'AlliedEsportsEntertainmentInc_20190815_8-K_EX-10.19_11788293_EX-10.19_Content License Agreement.pdf', 'blob_path': 'raw/license-agreement/a4a3ad8a-b3c8-4618-90d5-9d3c69b88bd8.pdf', 'file_size': 176502, 'mime_type': 'application/pdf', 'document_type': 'license-agreement', 'linked_entity': None, 'linked_entity_id': None, 'hash_sha256': '2c5fe2d4f89cd8f305e99eefac75177858c8fd57155b4347a3b010e270e3ae8a', 'text_extraction_status': 'ready', 'processing_status': 'pending extraction', 'acu_result_blob_path': None, 'created_at': datetime.datetime(2026, 2, 12, 6, 28, 52, 224803, tzinfo=datetime.timezone.utc), 'updated_at': datetime.datetime(2026, 2, 12, 6, 28, 52, 224803, tzinfo=datetime.timezone.utc)})
